# Download flat files
This was only possible from April 2024. We simply download everything and extract them to single ticker files.

In [1]:
import os
import pytz
import gzip
import calendar
import pandas as pd
from datetime import datetime, date, time, timedelta
from pytz import timezone
from times import get_market_dates
from fastparquet import write
import boto3
from botocore.config import Config
from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

session = boto3.Session(
   aws_access_key_id=ACCESS_KEY_ID,
   aws_secret_access_key=API_KEY,
)
s3 = session.client(
   's3',
   endpoint_url='https://files.polygon.io',
   config=Config(signature_version='s3v4'),
)

DATA_PATH = "../data/polygon/"

START_DATE = date(2016, 6, 1) # only go as far as your polygon membership allows. don't waste loop cycles
END_DATE = date(2016, 6, 11) # till today only or else you will get "out of range" from our custom function and don't waste loop cycles

# START_DATE = date(2016, 6, 1) # only go as far as your polygon membership allows. don't waste loop cycles

# till today only or else you will get "out of range" from our custom function and don't waste loop cycles
# this will need updating everyday so that we only fetch the new adjustments in an idempotent way
# END_DATE = date(2026, 6, 15)


# Initial download
Download everything

In [2]:
# for day in get_market_dates(START_DATE, END_DATE):
#     destination = DATA_PATH + f'raw/flatfiles/{day.isoformat()}.csv.gz'
#     s3.download_file('flatfiles', 
#                 f'us_stocks_sip/minute_aggs_v1/{day.year}/{day.strftime("%m")}/{day.isoformat()}.csv.gz', 
#                 destination)

In [3]:
df = pd.read_csv(DATA_PATH + f"raw/flatfiles/2016-06-06.csv.gz")
df

,ticker,volume,open,close,high,low,window_start,transactions
0,A,16484,45.72,45.780,45.78,45.72,1465219920000000000,35
1,A,1128,45.82,45.840,45.84,45.81,1465219980000000000,22
2,A,2637,45.81,45.845,45.88,45.81,1465220040000000000,31
3,A,953,45.84,45.800,45.84,45.80,1465220100000000000,18
4,A,4591,45.80,45.760,45.81,45.75,1465220160000000000,46
...,...,...,...,...,...,...,...,...
1263825,ZYNE,466,9.67,9.680,9.68,9.67,1465242840000000000,8
1263826,ZYNE,693,9.67,9.685,9.69,9.67,1465242960000000000,10
1263827,ZYNE,500,9.68,9.690,9.69,9.68,1465243020000000000,7
1263828,ZYNE,296,9.69,9.690,9.69,9.69,1465243080000000000,10


Extract and concatenate into monthly files

In [4]:
def concatenate_and_save(start_date, end_date, file_name):
    # Get a list of what we already have
    files = []
    # available_files = os.listdir(DATA_PATH + 'raw/flatfiles')
    # available_dates = [date.fromisoformat(file.replace(".csv.gz", "")) for file in available_files if file.endswith(".csv.gz")]
    
    market_dates = get_market_dates(start_date, end_date)
    if len(market_dates) == 0:
        return
    for day in market_dates:
        # if day >= start_date and day <= end_date and day in available_dates:
        destination = DATA_PATH + f'raw/flatfiles/{day.isoformat()}.csv.gz'
        with gzip.open(destination) as f:
            all_bars = pd.read_csv(f)
            all_bars = all_bars[['window_start', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'transactions']]
            all_bars = all_bars.rename(columns={'window_start': 'datetime'})
            all_bars = all_bars.set_index('datetime')
            all_bars.index = pd.to_datetime(all_bars.index, unit='ns') # Convert to datetime (UTC-naive)
            # Make UTC aware (in order to convert)
            # Convert UTC to ET
            # Make timezone naive
            all_bars.index = all_bars.index.tz_localize(pytz.UTC).tz_convert("US/Eastern").tz_localize(None)  

            files.append(all_bars)
            print(day)
        
    all_bars = pd.concat(files)
    all_bars = all_bars.reset_index()
    all_bars = all_bars.set_index('ticker')
    all_bars = all_bars.sort_index()

    all_bars.to_parquet(DATA_PATH + f"raw/flatfiles/monthly_parquet/{file_name}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

In [5]:
# for year in range(2016, 2016 + 1):
for year in range(2016, 2026 + 1):
    files = []
    # for month in range(6, 6 + 1):
    for month in range(1, 12 + 1):
        _, end_date = calendar.monthrange(year, month)
        concatenate_and_save(date(year, month, 1), date(year, month, end_date), f"{year}-{month}")

2016-06-06
2016-06-07
2016-06-08
2016-06-09
2016-06-10
2016-06-13
2016-06-14
2016-06-15
2016-06-16
2016-06-17
2016-06-20
2016-06-21
2016-06-22
2016-06-23
2016-06-24
2016-06-27
2016-06-28
2016-06-29
2016-06-30
2016-07-01
2016-07-05
2016-07-06
2016-07-07
2016-07-08
2016-07-11
2016-07-12
2016-07-13
2016-07-14
2016-07-15
2016-07-18
2016-07-19
2016-07-20
2016-07-21
2016-07-22
2016-07-25
2016-07-26
2016-07-27
2016-07-28
2016-07-29
2016-08-01
2016-08-02
2016-08-03
2016-08-04
2016-08-05
2016-08-08
2016-08-09
2016-08-10
2016-08-11
2016-08-12
2016-08-15
2016-08-16
2016-08-17
2016-08-18
2016-08-19
2016-08-22
2016-08-23
2016-08-24
2016-08-25
2016-08-26
2016-08-29
2016-08-30
2016-08-31
2016-09-01
2016-09-02
2016-09-06
2016-09-07
2016-09-08
2016-09-09
2016-09-12
2016-09-13
2016-09-14
2016-09-15
2016-09-16
2016-09-19
2016-09-20
2016-09-21
2016-09-22
2016-09-23
2016-09-26
2016-09-27
2016-09-28
2016-09-29
2016-09-30
2016-10-03
2016-10-04
2016-10-05
2016-10-06
2016-10-07
2016-10-10
2016-10-11
2016-10-12

In [6]:
df_parquet = pd.read_parquet(DATA_PATH + f"raw/flatfiles/monthly_parquet/2016-6.parquet")
df_parquet

,datetime,open,high,low,close,volume,transactions
ticker,,,,,,,
A,2016-06-06 09:32:00,45.7200,45.78,45.72,45.780,16484,35
A,2016-06-06 09:33:00,45.8200,45.84,45.81,45.840,1128,22
A,2016-06-06 09:34:00,45.8100,45.88,45.81,45.845,2637,31
A,2016-06-06 09:35:00,45.8400,45.84,45.80,45.800,953,18
A,2016-06-06 09:36:00,45.8000,45.81,45.75,45.760,4591,46
...,...,...,...,...,...,...,...
ZYNE,2016-06-30 15:55:00,6.8900,6.90,6.89,6.900,533,7
ZYNE,2016-06-30 15:56:00,6.8900,6.89,6.85,6.860,906,13
ZYNE,2016-06-30 15:57:00,6.8600,6.86,6.85,6.850,1000,11


Split the monthly files which contains all ticker into individual ticker files.

In [7]:
# for year in range(2016, 2016+1):
for year in range(2016, 2026+1):
    files = []
    # for month in range(6, 6+1):
    for month in range(1, 12+1):
        print(f'{datetime.now()} | {year}-{month}')
        if not os.path.isfile(DATA_PATH + f"raw/flatfiles/monthly_parquet/{year}-{month}.parquet"):
            continue
        
        all_bars = pd.read_parquet(DATA_PATH + f"raw/flatfiles/monthly_parquet/{year}-{month}.parquet")
        all_bars = all_bars[~all_bars.index.isna()]

        if all_bars['datetime'].min().year != year or all_bars['datetime'].min().month != month:
            raise Exception(f'{year} | {month} HAS OUT OF BOUND DATES')

        # File names are case insensitive! This lead to big data errors (e.g. TpC and TPC are merged)
        # So we simply remove all tickers that have small letters.
        # Which we don't need anyways, because small letter = non-common stock.
        for ticker in list(filter(lambda ticker: not any(s.islower() for s in ticker), list(all_bars.index.unique()))):
            bars = all_bars.loc[ticker]
            if isinstance(bars, pd.Series):
                bars = all_bars.loc[[ticker]]
            bars = bars[['datetime', 'open', 'high', 'low', 'close', 'volume', 'transactions']]
            bars = bars.set_index('datetime').sort_index()
            
            # Windows quirk note: you cannot save files called 'prn'. Of course there is a ticker that is named PRN...
            # So we name it 'PRN_'. I hope there are no tickers named NULL or something please.
            # DEV: NO NEED TO DO THIS FOR MAC.
            # if ticker == 'PRN':
            #     ticker = 'PRN_'

            if os.path.isfile(DATA_PATH + f'raw/m1/{ticker}.parquet'):
                write(DATA_PATH + f"raw/m1/{ticker}.parquet", bars, append=True, compression="snappy", row_group_offsets=25000)
            else:
                bars.to_parquet(DATA_PATH + f"raw/m1/{ticker}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

2026-06-07 13:02:31.623298 | 2016-1
2026-06-07 13:02:31.623526 | 2016-2
2026-06-07 13:02:31.623576 | 2016-3
2026-06-07 13:02:31.623594 | 2016-4
2026-06-07 13:02:31.623638 | 2016-5
2026-06-07 13:02:31.623765 | 2016-6
2026-06-07 13:03:04.369611 | 2016-7
2026-06-07 13:04:30.012004 | 2016-8
2026-06-07 13:06:07.309386 | 2016-9
2026-06-07 13:07:50.082866 | 2016-10
2026-06-07 13:09:39.106771 | 2016-11
2026-06-07 13:11:36.514693 | 2016-12
2026-06-07 13:13:52.099364 | 2017-1
2026-06-07 13:16:03.184448 | 2017-2
2026-06-07 13:18:10.391092 | 2017-3
2026-06-07 13:20:13.361591 | 2017-4
2026-06-07 13:22:26.882105 | 2017-5
2026-06-07 13:24:41.150172 | 2017-6
2026-06-07 13:27:01.051286 | 2017-7
2026-06-07 13:29:15.471859 | 2017-8
2026-06-07 13:31:51.387122 | 2017-9
2026-06-07 13:34:16.202088 | 2017-10
2026-06-07 13:36:41.971595 | 2017-11
2026-06-07 13:39:07.671749 | 2017-12
2026-06-07 13:41:32.860025 | 2018-1
2026-06-07 13:44:05.628518 | 2018-2
2026-06-07 13:46:33.382956 | 2018-3
2026-06-07 13:48:59.91

In [13]:
ticker_parquet = pd.read_parquet(DATA_PATH + f"raw/m1/AA.parquet")
ticker_parquet

,open,high,low,close,volume,transactions
datetime,,,,,,
2016-06-06 04:53:00,9.55,9.55,9.55,9.55,1000,2
2016-06-06 05:20:00,9.56,9.56,9.55,9.56,1200,5
2016-06-06 05:47:00,9.60,9.61,9.60,9.61,1380,13
2016-06-06 06:10:00,9.57,9.57,9.57,9.57,400,1
2016-06-06 06:54:00,9.60,9.60,9.60,9.60,100,1
...,...,...,...,...,...,...
2016-06-30 18:09:00,9.27,9.27,9.27,9.27,1550,1
2016-06-30 18:14:00,9.25,9.25,9.25,9.25,9040,4
2016-06-30 18:17:00,9.24,9.24,9.24,9.24,790,1


The old version of the above code runs very fast, up to end 2022. Then it becomes 20-50x slower. I found the problem:

Pandas dataframe lookup uses hash-tables so the time complexity is O(1). However, if you have null values in the index this does not apply anymore! The bars from end 2022 and upwards have null values.

# Updates
Process day-by-day

In [ ]:
def process_flatfile(local_file_path):
    """Unzips the flat file and split or append it to ticker files.
    """
    with gzip.open(local_file_path) as f:
        all_bars = pd.read_csv(f)
        all_bars = all_bars[['window_start', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'transactions']]
        all_bars = all_bars.rename(columns={'window_start': 'datetime'})
        all_bars = all_bars.set_index('datetime')
        all_bars.index = pd.to_datetime(all_bars.index, unit='ns') # Convert to datetime (UTC-naive)
        all_bars.index = all_bars.index.tz_localize(pytz.UTC)  # Make UTC aware (in order to convert)
        all_bars.index = all_bars.index.tz_convert("US/Eastern")  # Convert UTC to ET
        all_bars.index = all_bars.index.tz_localize(None)  # Make timezone naive
        
        for ticker in all_bars['ticker'].unique():
            bars = all_bars[all_bars['ticker'] == ticker]
            bars = bars[['open', 'high', 'low', 'close', 'volume', 'transactions']]

            if os.path.isfile(DATA_PATH + f'raw/m1/{ticker}.parquet'):
                write(DATA_PATH + f"raw/m1/{ticker}.parquet", bars, append=True, compression="snappy", row_group_offsets=25000)
            else:
                bars.to_parquet(DATA_PATH + f"raw/m1/{ticker}.parquet", engine="fastparquet", compression="snappy", row_group_offsets=25000)

In [ ]:
for day in get_market_dates(START_DATE, END_DATE):
    destination = DATA_PATH + f'raw/{day.isoformat()}.csv.gz' # DEV: THIS SEEMS WORNG! NO M1 OR FLATFILES FOLDER ??
    s3.download_file('flatfiles', 
                 f'us_stocks_sip/minute_aggs_v1/{day.year}/{day.strftime("%m")}/{day.isoformat()}.csv.gz', 
                 destination)
    process_flatfile(destination)